# MentorIA Tech — análise e previsão com telemetria real

**Notebook independente do backend.** Consulta o SQLite em **modo somente leitura**, não utiliza credenciais nem faz chamadas a Gemini/Groq. Os resultados deste notebook vêm exclusivamente do banco selecionado — **não gera dados sintéticos**.

**Objetivos:** auditar qualidade da telemetria; descrever consumo diário por operação/modelo; estimar taxas por usuário quando houver dados; treinar/validar previsão diária somente com histórico suficiente; apresentar um cenário de capacidade *condicional*, não uma previsão real validada de saturação.

> Importante: `telemetria_api` não identifica a **chave Gemini** utilizada. Portanto, não é possível atribuir o consumo observado à Chave 1 ou Chave 2 com fidelidade. O simulador de cotas abaixo é apenas um cenário configurável de **RPD**, não replica RPM/TPM nem o fallback real.

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Ajuste somente se o banco não estiver na mesma pasta do notebook.
DB = Path('novamente.db')
MIN_DIAS_TREINO = 60
HORIZONTE_DIAS = 30
CRESCIMENTO_USUARIOS_DIA = 0.03  # hipótese, não estimativa do banco
USUARIOS_INICIAIS = None  # None = último valor observado de usuários com geração

# Limites RPD ilustrativos: CONFIRME na sua conta antes de usar para decisões.
RPD = {'Gemini — Chave 1': 20, 'Gemini — Chave 2': 20, 'Groq': 1000}
SEED = 42


## 1. Leitura segura e verificação de esquema

In [ ]:
if not DB.is_file():
    raise FileNotFoundError(f'Banco não encontrado: {DB.resolve()}. Coloque o .ipynb junto de novamente.db ou altere DB.')

con = sqlite3.connect(DB.resolve().as_uri() + '?mode=ro', uri=True)
try:
    tabelas = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    obrigatorias = {'telemetria_api', 'geracoes'}
    if not obrigatorias.issubset(tabelas):
        raise ValueError(f'Tabelas ausentes: {sorted(obrigatorias - tabelas)}')
    colunas = {t: {r[1] for r in con.execute(f'PRAGMA table_info({t})')} for t in obrigatorias}
    esperadas = {
        'telemetria_api': {'data_hora','provedor','modelo','operacao','tokens_totais','codigo_http','status','geracao_id'},
        'geracoes': {'data_hora','usuario_id','geracao_id'}
    }
    for tabela, cols in esperadas.items():
        if not cols.issubset(colunas[tabela]):
            raise ValueError(f'{tabela}: colunas ausentes {sorted(cols - colunas[tabela])}')
    chamadas = pd.read_sql_query('''SELECT data_hora, provedor, modelo, operacao, tokens_totais,
                                            codigo_http, status, geracao_id
                                     FROM telemetria_api''', con)
    geracoes = pd.read_sql_query('''SELECT geracao_id, data_hora, usuario_id FROM geracoes''', con)
finally:
    con.close()

for df in (chamadas, geracoes):
    df['instante'] = pd.to_datetime(df['data_hora'], utc=True, errors='coerce')
    df.dropna(subset=['instante'], inplace=True)
    df['dia'] = df['instante'].dt.floor('D').dt.tz_localize(None)

print(f'Chamadas: {len(chamadas)} | Gerações: {len(geracoes)}')
print(f'Dias com chamadas: {chamadas.dia.nunique()} | Dias com gerações: {geracoes.dia.nunique()}')
print('Datas em UTC; o agrupamento diário segue UTC, não o fuso local.')


## 2. Qualidade da telemetria e consumo por modelo

In [ ]:
qualidade = pd.DataFrame([{
    'chamadas':len(chamadas),
    'tokens_informados':int(chamadas.tokens_totais.notna().sum()),
    'tokens_ausentes':int(chamadas.tokens_totais.isna().sum()),
    'erros_http_429':int(chamadas.codigo_http.eq(429).sum()),
    'chamadas_sem_geracao_id':int(chamadas.geracao_id.isna().sum()),
    'modelos':int(chamadas.modelo.nunique())
}])
display(qualidade)
por_modelo = (chamadas.groupby(['provedor','modelo','operacao'],dropna=False)
              .agg(chamadas=('data_hora','size'), tokens_conhecidos=('tokens_totais','sum'),
                   com_tokens=('tokens_totais','count'), erros_429=('codigo_http',lambda x: int(x.eq(429).sum())))
              .reset_index().sort_values('chamadas',ascending=False))
display(por_modelo)
print('Atenção: tokens_conhecidos soma apenas registros preenchidos; não equivale necessariamente ao consumo total.')


## 3. Série diária observada e usuários com geração registrada

A contagem de usuários é um **proxy**: inclui apenas usuários presentes em `geracoes`. Uma chamada de API pode ser uma tentativa de fallback ou uma operação auxiliar; não corresponde necessariamente a uma solicitação original.

In [ ]:
if chamadas.empty:
    raise ValueError('Não há chamadas com datas válidas para analisar.')

por_dia = (chamadas.groupby('dia').agg(
    chamadas=('data_hora','size'), tokens_conhecidos=('tokens_totais','sum'),
    com_tokens=('tokens_totais','count'), erros_429=('codigo_http',lambda x:int(x.eq(429).sum()))
))
usuarios = geracoes.groupby('dia').usuario_id.nunique().rename('usuarios_com_geracao')
inicio = min(chamadas.dia.min(), geracoes.dia.min() if not geracoes.empty else chamadas.dia.min())
fim = max(chamadas.dia.max(), geracoes.dia.max() if not geracoes.empty else chamadas.dia.max())
calendario = pd.date_range(inicio,fim,freq='D')
diario = por_dia.reindex(calendario,fill_value=0).join(usuarios,how='left')
diario['usuarios_com_geracao'] = diario['usuarios_com_geracao'].fillna(0).astype(int)
# Dias sem chamada são zeros apenas DENTRO do intervalo observado. Dias após o último registro não são preenchidos.
diario['cobertura_tokens_%'] = np.where(diario.chamadas>0,100*diario.com_tokens/diario.chamadas,np.nan)
diario['chamadas_por_usuario_proxy'] = (diario.chamadas / diario.usuarios_com_geracao.replace(0,np.nan))
diario.index.name='data_utc'
display(diario.tail(15).round(2))
fig, axes = plt.subplots(2,1,figsize=(11,6),sharex=True)
diario[['chamadas','erros_429']].plot(ax=axes[0],title='Chamadas e erros 429 — observados')
diario[['tokens_conhecidos']].plot(ax=axes[1],title='Tokens registrados (limite inferior quando há ausências)')
plt.tight_layout(); plt.show()


## 4. Verificação de suficiência e validação cronológica

O treinamento é **bloqueado** quando houver menos de 60 dias completos de observação. Mesmo após 60 dias, os resultados devem ser examinados quanto a mudanças de instrumentação, dias sem telemetria e representatividade. O último dia é excluído por poder estar incompleto.

In [ ]:
# Evita considerar o dia mais recente como completo, mesmo que pareça ter todas as chamadas.
dias_completos = diario.iloc[:-1].copy()
quantidade = len(dias_completos)
print(f'Dias de calendário antes do último dia (potencialmente incompleto): {quantidade}')
PODE_TREINAR = quantidade >= MIN_DIAS_TREINO and (dias_completos.chamadas>0).sum() >= 30
if not PODE_TREINAR:
    print('TREINAMENTO NÃO EXECUTADO: histórico insuficiente para validação temporal minimamente útil.')
    print('Os gráficos acima são observações reais; não há métricas preditivas reais a apresentar ainda.')
else:
    print('Histórico mínimo atingido. Iniciando avaliação cronológica exploratória.')


## 5. Modelo C adaptado para dados reais (executa somente quando permitido)

Aprende **chamadas por usuário com geração**, usando dia da semana e consumo recente. Reconstrói chamadas totais multiplicando pela quantidade observada de usuários no teste. **Isso é uma avaliação condicional**: em produção também será necessário prever o número de usuários. Para tokens, usamos uma taxa histórica de tokens por chamada com cobertura informada; não imputamos tokens ausentes como zero.

In [ ]:
if PODE_TREINAR:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, r2_score

    base = dias_completos.copy()
    base['dia_semana'] = base.index.dayofweek
    base['fim_semana'] = (base.dia_semana>=5).astype(int)
    base['taxa_chamadas'] = base.chamadas / base.usuarios_com_geracao.replace(0,np.nan)
    # Taxa defasada: só usa informação disponível antes do dia a prever.
    base['taxa_lag1'] = base.taxa_chamadas.shift(1)
    base['taxa_media7'] = base.taxa_chamadas.shift(1).rolling(7,min_periods=3).mean()
    cols = ['dia_semana','fim_semana','taxa_lag1','taxa_media7']
    treino_dados = base.dropna(subset=cols+['taxa_chamadas']).copy()
    if len(treino_dados) < 40:
        print('TREINAMENTO NÃO EXECUTADO: poucos dias com usuários identificáveis e taxas válidas.')
        PODE_TREINAR = False
    else:
        corte = int(len(treino_dados)*0.8)
        tr, te = treino_dados.iloc[:corte], treino_dados.iloc[corte:]
        modelo = RandomForestRegressor(n_estimators=150,min_samples_leaf=3,random_state=SEED,n_jobs=-1)
        modelo.fit(tr[cols],tr.taxa_chamadas)
        previsto = np.maximum(0,modelo.predict(te[cols]))*te.usuarios_com_geracao.to_numpy()
        observado = te.chamadas.to_numpy()
        baseline = tr.taxa_chamadas.mean()*te.usuarios_com_geracao.to_numpy()
        def resumo(nome, yhat):
            return {'modelo':nome,'MAE_chamadas':mean_absolute_error(observado,yhat),
                    'WAPE_%':100*np.abs(observado-yhat).sum()/max(1,observado.sum()),
                    'R2':r2_score(observado,yhat) if len(observado)>1 else np.nan,
                    'vies':np.mean(yhat-observado)}
        display(pd.DataFrame([resumo('RF por usuário (condicional)',previsto),resumo('Baseline taxa média',baseline)]).round(3))
        ax=pd.DataFrame({'observado':observado,'RF':previsto,'baseline':baseline},index=te.index).plot(figsize=(11,4),title='Teste cronológico — chamadas/dia')
        plt.show()
        print('IMPORTANTE: o teste utiliza o número REAL de usuários do próprio dia, conhecido retrospectivamente.')


## 6. Cenário futuro exploratório — não é previsão validada

Com histórico curto, é possível calcular apenas uma **projeção condicional simples**, baseada na taxa observada de chamadas por usuário. Não chamamos esse cenário de previsão do Random Forest. As cotas RPD reiniciam diariamente, e as duas chaves Gemini são tratadas separadamente, mas sem associação real de chave no SQLite.

In [ ]:
taxas_validas = diario.chamadas_por_usuario_proxy.replace([np.inf,-np.inf],np.nan).dropna()
if taxas_validas.empty:
    print('Cenário indisponível: não há dias com chamadas e usuários com geração registrados simultaneamente.')
else:
    taxa = float(taxas_validas.median())
    ultimos_usuarios = diario.loc[diario.usuarios_com_geracao>0,'usuarios_com_geracao']
    if ultimos_usuarios.empty and USUARIOS_INICIAIS is None:
        print('Defina USUARIOS_INICIAIS para executar o cenário.')
    else:
        u0 = int(USUARIOS_INICIAIS if USUARIOS_INICIAIS is not None else ultimos_usuarios.iloc[-1])
        datas = pd.date_range(diario.index.max()+pd.Timedelta(days=1),periods=HORIZONTE_DIAS,freq='D')
        usuarios_futuros = np.maximum(1,np.rint(u0*(1+CRESCIMENTO_USUARIOS_DIA)**np.arange(1,HORIZONTE_DIAS+1))).astype(int)
        solicitadas = np.rint(usuarios_futuros*taxa).astype(int)
        # Cenário de RPD agregado apenas. Não representa quotas por modelo/operação nem picos RPM/TPM.
        capacidade = sum(RPD.values())
        recusadas = np.maximum(0,solicitadas-capacidade)
        cenario = pd.DataFrame({'usuarios_hipoteticos':usuarios_futuros,'chamadas_estimadas':solicitadas,
                                'capacidade_RPD_ilustrativa':capacidade,'recusadas_por_RPD':recusadas},index=datas)
        cenario['recusa_%'] = 100*recusadas/np.maximum(solicitadas,1)
        display(cenario.head(10).round(2))
        critico = cenario[cenario['recusa_%']>=5]
        print('Primeiro dia >=5% recusas no cenário RPD:' , critico.index[0].date() if len(critico) else 'não observado')
        ax=cenario[['chamadas_estimadas','capacidade_RPD_ilustrativa']].plot(figsize=(11,4),title='Cenário condicional de demanda e RPD (não validado)')
        plt.show()
        print('AVISO: as chamadas não são intercambiáveis entre modelos; esta capacidade somada é apenas um limite ilustrativo.')


## 7. O que falta para uma previsão real de saturação

1. **Mais histórico representativo:** semanas ou meses de uso, com registro contínuo e sem dias artificiais de zero.
2. **Identificar a chave/projeto e o modelo em cada tentativa:** sem isso não é possível reproduzir fielmente o consumo de Gemini — Chave 1, Gemini — Chave 2 e Groq.
3. **Medir tokens em todas as operações possíveis**, documentando os registros ausentes e distinguindo tentativas, chamadas lógicas e fallback.
4. **Registrar/estimar usuários ativos futuros**; a validação acima usa o número observado de usuários no dia de teste, que não estaria disponível antecipadamente.
5. **Calibrar cotas por modelo e operação**, incluindo RPM, TPM, RPD e demais limites relevantes; testar o roteamento real.
6. **Validação temporal de ponta a ponta:** comparar datas previstas de saturação com eventos observados, medir recall, precisão, falsos alarmes e erro em dias, inclusive quando não ocorre saturação.

**Publicação:** distribua o notebook e instruções de execução, mas **não** publique o banco real, chaves de API, identificadores de usuários nem saídas sensíveis. Limpe as saídas antes de fazer commit.
